In [ ]:
# !pip install pandas

In [1]:
import numpy as np

# tensorflow: 이번 노트북의 주인공입니다.
#   - tf.keras.Sequentail: 모델을 쌓아 만드는 고수준 API
#   - tf.keras.layers.Dense: H = a * X_norm + b를 계산하는 선형 부품 (PyTorch의 Linear에 대응)
#   - tf.keras.losses.BinaryCrossentropy(from_logits=True): H를 받아 sigmoid+BCE를 내부 처리 (BCEWithLogitsLoss에 대응)
#   - tf.keras.optimizers.SGD: 파라미터 업데이트 (PyTorch의 SGD optimizer에 대응)
import tensorflow as tf

import pandas as pd

In [2]:
# seed 고정: 실행할 때마다 초기 weight가 비슷해져 결과를 비교하기 쉽습니다.
#   - np.random.seed(42)      : NumPy 쪽 난수 고정
#   - tf.random.set_seed(42)  : TensorFlow 쪽 난수 고정 (Dense layer 초기 weight에 영향)
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow 버전:', tf.__version__)
print('NumPy 버전:', np.__version__)
print('Pandas 버전:', pd.__version__)

TensorFlow 버전: 2.10.0
NumPy 버전: 1.23.5
Pandas 버전: 2.3.3


In [3]:
# H는 sigmoid 이전의 선형 출력값입니다.
# H는 확률이 아니므로 음수, 0, 양수, 1보다 큰 값이 모두 가능합니다.
# H는 선형 계산 결과이므로 -3, -1, 0, 1, 3처럼 다양한 실수값이 될 수 있습니다.
# (앞 셀에서 import한 numpy(np), tensorflow(tf), pandas(pd)를 그대로 사용합니다.)

# 여기서는 H 값을 직접 만든 뒤 sigmoid를 적용해 보기 위해 NumPy 배열을 사용합니다.
# 아직 모델을 학습하는 단계가 아니므로 TensorFlow Tensor로 만들 필요는 없습니다.

# shape을 (5, 1)로 만든 이유:
# - 데이터가 5개 있습니다.
# - 각 데이터는 H 값 1개를 가집니다.
# - 따라서 모양은 (데이터 개수, 1) = (5, 1)입니다.
H_example = np.array([[-3.0], [-1.0], [0.0], [1.0], [3.0]])

H_example

array([[-3.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 3.]])

In [4]:
# z는 sigmoid(H)로 얻는 값입니다.
# z는 항상 0과 1 사이의 값이므로 예측 확률로 해석할 수 있습니다.

# [NumPy 배열 -> tf.sigmoid -> TensorFlow Tensor]
#   H_example은 NumPy 배열입니다.
#   tf.sigmoid()는 NumPy 배열도 입력으로 받을 수 있습니다.
#   tf.sigmoid(H_example)을 적용하면 TensorFlow가 이 값을 받아 계산하고,
#   그 결과인 z_example은 TensorFlow Tensor로 반환됩니다.
z_example = tf.sigmoid(H_example)

z_example

<tf.Tensor: shape=(5, 1), dtype=float64, numpy=
array([[0.04742587],
       [0.26894142],
       [0.5       ],
       [0.73105858],
       [0.95257413]])>

In [5]:
# H와 z를 표로 나란히 놓고 비교합니다. (pandas DataFrame 사용)

# [pd.DataFrame이란?]
#   pd.DataFrame은 값을 '표(table)' 형태로 보여 주기 위한 도구입니다.
#   모델 학습에 꼭 필요한 코드는 아니지만, H와 z를 나란히 비교하면
#   'H는 확률이 아니고, z가 확률이다'라는 점을 눈으로 확인하기 쉽습니다.

# [reshape(-1) / .numpy()가 왜 필요한가?]
#   H_example, z_example은 shape이 (5, 1)인 2차원(5행 1열) 데이터입니다.
#   DataFrame의 한 '열(column)'로 넣으려면 [값1, 값2, 값3, ...] 형태의 1차원 배열이 편합니다.
#     - .reshape(-1)  : (5, 1) 같은 2차원 모양을 (5,) 1차원으로 '펴 줍니다'. (-1은 길이를 알아서 맞추라는 뜻)
#     - .numpy()      : TensorFlow Tensor를 NumPy 배열로 바꿉니다. (Tensor에만 필요)
sigmoid_result_df = pd.DataFrame({
    # H_example은 NumPy 배열이므로 .numpy()가 필요 없습니다.
    # reshape(-1)은 (5, 1) 모양을 DataFrame 열에 넣기 쉬운 1차원 형태로 펴는 역할입니다.
    'H': H_example.reshape(-1),
    
    # z_example은 TensorFlow Tensor이므로 .numpy()로 NumPy 배열로 꺼냅니다.
    'z = sigmoid(H)': z_example.numpy().reshape(-1)    # sigmoid를 통과한 예측 확률 (항상 0~1 사이)
})

# sigmoid 함수는 H를 0과 1 사이의 값으로 바꿉니다.
# 따라서 sigmoid(H)의 결과인 z를 예측 확률로 해석합니다.
#   정리: H = sigmoid 이전의 선형 출력값
#         z = sigmoid(H) = 예측 확률
sigmoid_result_df

,H,z = sigmoid(H)
0,-3.0,0.047426
1,-1.0,0.268941
2,0.0,0.500000
3,1.0,0.731059
4,3.0,0.952574


In [6]:
# 이진 분류에서는 예측 확률 z를 기준으로 최종 클래스를 결정합니다.
# 이번 입문 예제에서는 기본 기준값으로 0.5를 사용합니다.
#   z >= 0.5 이면 1
#   z <  0.5 이면 0

# [tf.cast(z_example >= 0.5, tf.int32) 을 초보자용으로 풀어 보면]
#   1) z_example >= 0.5 -> 각 원소가 조건을 만족하는지에 따라 True 또는 False 값을 만듭니다.
#   2) tf.cast(..., tf.int32) -> True를 1로, False를 0으로 '형 변환(cast)'합니다.
#   즉, 확률 z를 최종 분류값 0 또는 1로 바꾸는 코드입니다.
prediction_example = tf.cast(z_example >= 0.5, tf.int32)

# 앞에서 만든 표(sigmoid_result_df)에 prediction 열을 추가해 H, z, 최종 분류를 한 눈에 봅니다.
# (여기서도 .numpy().reshape(-1)로 Tensor를 1차원 NumPy 배열로 펴서 표의 한 열로 넣습니다.)
sigmoid_result_df['prediction'] = prediction_example.numpy().reshape(-1)

sigmoid_result_df

,H,z = sigmoid(H),prediction
0,-3.0,0.047426,0
1,-1.0,0.268941,0
2,0.0,0.500000,1
3,1.0,0.731059,1
4,3.0,0.952574,1


In [7]:
# 3. Sequential 모델 생성 (Dense(1) = PyTorch의 Linear(1, 1))

# tf.keras.Input(shape=(1,)) : 입력이 '데이터 한 개당 특성 1개'라는 것을 알려 줍니다.
#                              (데이터 개수가 아니라 '특성 개수'라는 점에 주의!)
# tf.keras.layers.Dense(1)   : H = a * X_norm + b를 계산하는 선형 층입니다.
# (중요) activation을 넣지 않습니다! 따라서 출력은 확률 z가 아니라 H(선형 계산값)입니다.
#       (sigmoid는 뒤에서 BinaryCrossentropy(from_logits=True)가 내부 처리합니다.)

# 모델을 만들기 직전에 seed를 한 번 더 고정합니다.
#   주의: tf.constant나 tf.sigmoid는 난수를 사용하는 연산이 아닙니다.
#         따라서 위 '1) sigmoid 복습' 준비 실습 자체가 Dense 초기 weight를 바꾸는 원인은 아닙니다.
#   그렇다면 왜 다시 고정할까요?
#     - 교육용 
model = tf.keras.Sequential([
            tf.keras.Input(shape=(1,)),
            tf.keras.layers.Dense(1)
        ])

# 모델 구조를 요약해서 봅니다.
#   아래 출력에서 Param # 가 2로 보이는 이유는
#   Dense(1)이 weight a 1개 + bias b 1개 = 총 2개의 파라미터를 가지기 때문입니다.
#   즉, 이 모델이 학습으로 바꾸는 값은 a와 b 두 개뿐입니다.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________


In [9]:
# 1. 입력값 X와 정답 y 준비 (이전 PyTorch 노트북과 동일한 값)

# X: 입력값(사람의 키 cm). dtype=np.float32로 실수 형식, reshape(-1, 1)로 (n, 1) 형태.
#    Dense(1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 (n, 1) 형태여야 합니다.

#    [reshape(-1, 1)을 초보자용으로 더 자세히]
#      Dense(1)은 입력을 '표(table)' 형태로 받는다고 생각하면 쉽습니다.
#        - 행(row)    = 데이터 개수 (사람 수). 여기서는 키가 4개이므로 4행.
#        - 열(column) = 입력 특성 개수. 이번 예제는 '키' 하나만 쓰므로 특성은 1개 -> 1열.
#      따라서 X의 shape은 (데이터 개수, 1) = (4, 1)이 되어야 합니다.
#      reshape(-1, 1) 에서:
#        -1 : 행(데이터 개수)은 NumPy가 알아서 계산하라는 뜻. (여기서는 4로 자동 결정됨)
#         1 : 각 데이터가 입력 특성 1개를 가진다는 뜻.
X = np.array([160, 170, 180, 190], dtype=np.float32).reshape(-1, 1)

# y: 정답값(0=농구선수 아님, 1=농구선수). 마찬가지로 (n, 1) 형태로 맞춥니다.
y = np.array([0, 0, 1, 1], dtype=np.float32).reshape(-1, 1)

print('입력값 X:\n', X)
print('정답값 y:\n', y)
print('X shape:', X.shape)
print('y shape:', y.shape)

입력값 X:
 [[160.]
 [170.]
 [180.]
 [190.]]
정답값 y:
 [[0.]
 [0.]
 [1.]
 [1.]]
X shape: (4, 1)
y shape: (4, 1)


In [11]:
# 2. 입력값 정규화 (이전 파일과 동일)

# 평균과 표준편차를 계산
# - 평균: 데이터의 중심(가운데쯤 되는 값)
# - 표준편차: 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 입력값의 범위를 0 근처로 비슷하게 맞추면 학습이 더 안정적으로 진행됨
# 주의: 실제 학습에는 원래 키 X가 아니라, 정규화된 입력값 X_norm을 사용
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:\n', X_norm)
print('X_norm shape:', X_norm.shape)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.18034
정규화된 입력값 X_norm:
 [[-1.3416408]
 [-0.4472136]
 [ 0.4472136]
 [ 1.3416408]]
X_norm shape: (4, 1)


In [12]:
# 3. 학습 전 model 출력 확인 (출력은 확률이 아니라 H)

# model(X_norm): 이번 구조에서는 H(선형 계산값)를 반환합니다. (확률 아님)
H_before = model(X_norm)

# 예측 확률 z를 보려면 H에 sigmoid를 직접 적용해야 합니다. (z = sigmoid(H))
z_before = tf.sigmoid(H_before)

print('학습 전 H 예시:')
print(H_before.numpy()[:5])

print('\n학습 전 sigmoid(H), 즉 z 예시:')
print(z_before.numpy()[:5])

# H와 z는 서로 다른 값입니다. H는 확률이 아니고, z(=sigmoid(H))가 0~1 사이의 확률입니다.

학습 전 H 예시:
[[-0.9208435 ]
 [-0.30694783]
 [ 0.30694783]
 [ 0.9208435 ]]

학습 전 sigmoid(H), 즉 z 예시:
[[0.28478605]
 [0.4238599 ]
 [0.57614005]
 [0.7152139 ]]
